# BODAQS Event Schema Test Harness

Purpose: rapid schema tuning for a single CSV.

Edit `SCHEMA_PATH` and `CSV_PATH` below, then Run All.


In [1]:
CSV_PATH = r"logs_test\2026-02-20_13-08-45.CSV"
SCHEMA_PATH = r"event schema\event_schema_tests.yaml"

NORMALIZE_RANGES = {
    "front_shock_dom_suspension [mm]": 170.0,
    "rear_shock_dom_suspension [mm]": 150.0,
}

ZEROING_ENABLED = False
SAMPLE_RATE_HZ = None
BUTTERWORTH_SMOOTHING = []  # e.g. [{"cutoff_hz": 3.0, "order": 4}]
BUTTERWORTH_GENERATE_RESIDUALS = False
INGESTION_MODE = "tolerant"  # "tolerant" or "strict"

ACTIVE_SIGNAL_DISP_COL = None  # e.g. "front_shock_dom_suspension [mm]"
ACTIVE_DISP_THRESH = 20.0
ACTIVE_VEL_THRESH = 50.0
ACTIVE_WINDOW = "500ms"
ACTIVE_PADDING = "1s"
ACTIVE_MIN_SEG = "3s"

CLIP_0_1 = False
RUN_TZ_LABEL = "AWST"


In [2]:
from pathlib import Path
import logging

import pandas as pd

from bodaqs_analysis.pipeline import run_macro
from bodaqs_analysis.artifacts import (
    ArtifactStore,
    copy_raw_csv_to_source,
    ensure_run_is_new,
    ensure_session_is_new,
    make_run_id,
    save_session_artifacts,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    write_run_manifest,
    write_session_manifest,
)

csv_path = Path(CSV_PATH)
schema_path = Path(SCHEMA_PATH)

if not csv_path.exists():
    raise FileNotFoundError(f"CSV_PATH does not exist: {csv_path}")
if not schema_path.exists():
    raise FileNotFoundError(f"SCHEMA_PATH does not exist: {schema_path}")
if INGESTION_MODE not in {"tolerant", "strict"}:
    raise ValueError("INGESTION_MODE must be 'tolerant' or 'strict'.")
if not NORMALIZE_RANGES:
    raise ValueError("Set NORMALIZE_RANGES = {'signal [unit]': full_range, ...} before running this cell.")

root = logging.getLogger()
root.setLevel(logging.WARNING)
for handler in root.handlers:
    handler.setLevel(logging.WARNING)

store = ArtifactStore(Path("artifacts"))
run_id_base = make_run_id(tz_label=RUN_TZ_LABEL)
run_id = run_id_base
suffix = 1
while store.run_dir(run_id).exists():
    run_id = f"{run_id_base}_{suffix:02d}"
    suffix += 1
ensure_run_is_new(store, run_id=run_id, force=False)

results = run_macro(
    str(csv_path),
    str(schema_path),
    zeroing_enabled=ZEROING_ENABLED,
    zero_window_s=0.4,
    zero_min_samples=10,
    clip_0_1=CLIP_0_1,
    active_signal_disp_col=ACTIVE_SIGNAL_DISP_COL,
    active_signal_vel_col=None,
    active_disp_thresh=ACTIVE_DISP_THRESH,
    active_vel_thresh=ACTIVE_VEL_THRESH,
    active_window=ACTIVE_WINDOW,
    active_padding=ACTIVE_PADDING,
    active_min_seg=ACTIVE_MIN_SEG,
    normalize_ranges=NORMALIZE_RANGES,
    sample_rate_hz=SAMPLE_RATE_HZ,
    butterworth_smoothing=BUTTERWORTH_SMOOTHING,
    butterworth_generate_residuals=BUTTERWORTH_GENERATE_RESIDUALS,
    strict=(INGESTION_MODE == "strict"),
)

session = results["session"]
session_id = str(session["session_id"])
schema = results.get("schema")
events_df = results.get("events", pd.DataFrame())
metrics_df = results.get("metrics", pd.DataFrame())

ensure_session_is_new(store, run_id=run_id, session_id=session_id, force=False)

source_sha256 = copy_raw_csv_to_source(
    store=store,
    run_id=run_id,
    session_id=session_id,
    csv_path=csv_path,
)

save_session_artifacts(
    store,
    run_id=run_id,
    session_id=session_id,
    session_df=session["df"],
    session_meta=session["meta"],
)

write_session_manifest(
    store,
    run_id=run_id,
    session_id=session_id,
    contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
    source={"path": "source/input.csv", "sha256": source_sha256},
    summary={
        "n_rows": int(len(session["df"])),
        "n_events": int(len(events_df)) if isinstance(events_df, pd.DataFrame) else 0,
        "n_metrics": int(len(metrics_df)) if isinstance(metrics_df, pd.DataFrame) else 0,
    },
)

if isinstance(events_df, pd.DataFrame) and not events_df.empty:
    write_events_partitioned_by_schema_id(
        store=store,
        run_id=run_id,
        session_id=session_id,
        events_df=events_df,
        schema_path=schema_path,
    )

if isinstance(metrics_df, pd.DataFrame) and not metrics_df.empty:
    write_metrics_partitioned_by_schema_id(
        store=store,
        run_id=run_id,
        session_id=session_id,
        metrics_df=metrics_df,
    )

write_run_manifest(
    store,
    run_id=run_id,
    session_ids=[session_id],
    timezone_label=RUN_TZ_LABEL,
    pipeline_config={
        "schema_path": str(schema_path),
        "zeroing_enabled": bool(ZEROING_ENABLED),
        "normalize_ranges": bool(NORMALIZE_RANGES),
        "butterworth_smoothing": bool(BUTTERWORTH_SMOOTHING),
        "butterworth_generate_residuals": bool(BUTTERWORTH_GENERATE_RESIDUALS),
        "sample_rate_hz": SAMPLE_RATE_HZ,
        "ingestion_mode": INGESTION_MODE,
        "n_files": 1,
    },
)

session_key = f"{run_id}::{session_id}"
artifact_session_dir = store.session_dir(run_id, session_id)

print(f"session_id: {session_id}")
print(f"run_id: {run_id}")
print(f"artifact_session_dir: {artifact_session_dir}")
print(f"events: {len(events_df) if isinstance(events_df, pd.DataFrame) else 0}")
print(f"metrics: {len(metrics_df) if isinstance(metrics_df, pd.DataFrame) else 0}")


session_id: 2026-02-20_13-08-45
run_id: run_2026-03-15T16-47-42_AWST
artifact_session_dir: artifacts\runs\run_2026-03-15T16-47-42_AWST\sessions\2026-02-20_13-08-45
events: 33
metrics: 33


In [3]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader
from bodaqs_analysis.widgets.loaders import make_session_loader, load_all_events_for_selected
from IPython.display import display

if "store" not in globals() or "session_key" not in globals():
    raise RuntimeError("Run the preprocessing cell first.")

key_to_ref = {session_key: (run_id, session_id)}
session_loader = make_session_loader(store=store, key_to_ref=key_to_ref)
events_df_sel = load_all_events_for_selected(store, key_to_ref=key_to_ref)

if events_df_sel.empty:
    raise ValueError(
        "No events were written for this session. Check SCHEMA_PATH, NORMALIZE_RANGES, and the preprocessing output."
    )

browser = make_event_browser_widget_for_loader(
    schema,
    events_df_sel,
    session_loader=session_loader,
    session_key_col="session_key",
)

display(browser["ui"])


## Iteration tips

- Edit the schema YAML, then re-run the `Run preprocessing` cell and the `Event browser` cell.
- This harness writes artifacts under `analysis/artifacts/runs/<run_id>/sessions/<session_id>/`.
- It does not use the file-selector SHA cache in `.bodaqs_preprocess_sha_cache.json`; each preprocess run creates a fresh `run_id`.
- If you want a clean rerun, run the preprocessing cell again to generate a new run folder, or delete the old run folder before re-running.
- The event browser keeps an in-memory session cache for the current widget instance; re-running the browser cell rebuilds it against the latest processed session.
